# DSC@UIT 2026 — Task 2: fine-tune Qwen3.5-2B evidence selector

## Model được chọn

Dùng **Qwen/Qwen3.5-2B**. Model ID này có trong danh sách mô hình BTC, trang 3 của PDF. Notebook Task 2 hiện tại cũng đã khai báo model này cho nhánh S18 để chọn chứng cứ; adapter này fine-tune đúng vai trò đó.

Không để LLM tự viết câu trả lời cuối. Model chỉ chọn tối đa 3 mục trong 8 căn cứ ứng viên; pipeline hiện tại tiếp tục ghép câu trả lời bằng văn bản pháp luật trích từ corpus. Cách này giữ câu trả lời bám nguồn và khớp cấu trúc S18 hơn RAG sinh tự do.

## Cách fine-tune

- Chỉ dùng train.json và corpus chính thức của Task 2.
- Tách FIT/DEV theo nhóm câu hỏi đã chuẩn hóa và citation trùng nhau.
- Với mỗi câu FIT, truy hồi 8 căn cứ từ corpus. Tạo nhãn bạc tự động bằng greedy search: chọn tập căn cứ có METEOR cao nhất so với answer chính thức. Không có nhãn tay.
- LoRA trên model gốc, loss chỉ trên token chọn số thứ tự căn cứ. Chọn checkpoint theo METEOR sinh trên DEV.
- So sánh base và adapter trên cùng DEV trước khi bật adapter trong pipeline chính. Metric trong notebook là proxy trên phần văn bản căn cứ đã chọn; điểm quyết định cuối vẫn là METEOR end-to-end của pipeline chính thức.

Task 1 không dùng adapter này: Task 1 xếp hạng ID tài liệu và bi-encoder/cross-encoder hiện tại phù hợp hơn với Recall@5.

## Điều kiện cuộc thi

Model có trong PDF chưa đồng nghĩa đội đã đăng ký và được BTC phê duyệt model đó. Thời hạn đăng ký trong tài liệu cuộc thi kết thúc ngày 18/09/2026. Chỉ đặt TEAM_MODEL_APPROVED=True nếu model ID chính xác này đã nằm trong danh sách được duyệt cho đội; nếu chưa, không đưa adapter vào bài nộp.

Task 2 đang có khoảng 1.136B tham số embedding + reranker. Cộng Qwen3.5-2B là khoảng 3.136B trước adapter; notebook đếm tham số gốc + LoRA và kiểm tra ngưỡng 4B. LoRA, FP16 và lưu adapter không làm giảm số tham số bị tính.


In [1]:
#@title Cài dependency huấn luyện
# Cài Transformers có hỗ trợ Qwen3.5; chỉ dùng bản Transformers chính thức từ Hugging Face.
import os
# Reduce CUDA allocator fragmentation before torch creates CUDA memory.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess, sys

_TRANSFORMERS_LOADED_BEFORE_SETUP = "transformers" in sys.modules
_SAFETENSORS_LOADED_BEFORE_SETUP = "safetensors" in sys.modules

def _pip_install(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages],
        check=True,
    )

def _probe(code):
    return subprocess.run(
        [sys.executable, "-c", code], capture_output=True, text=True
    )

_pip_install("peft", "accelerate", "scikit-learn", "nltk")

# Kaggle đôi khi cài sẵn torchao cũ. LoRA FP16 tiêu chuẩn không dùng torchao,
# còn PEFT mới có thể dừng vì phiên bản tùy chọn này quá cũ.
_torchao_probe_code = (
    "import importlib.util; "
    "raise SystemExit(0 if importlib.util.find_spec('torchao') is not None else 1)"
)
_torchao_check = _probe(_torchao_probe_code)
if _torchao_check.returncode == 0:
    if "torchao" in sys.modules or "peft" in sys.modules:
        raise RuntimeError(
            "torchao/PEFT đã được nạp trong phiên hiện tại. Hãy Restart Session rồi chạy lại từ đầu."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
        check=True,
    )
    _torchao_check = _probe(_torchao_probe_code)
    if _torchao_check.returncode == 0:
        raise RuntimeError("Không gỡ được torchao cũ khỏi môi trường Kaggle.")
print("torchao: không dùng; cấu hình LoRA FP16 dùng Linear mặc định của PEFT.")

_safe_code = (
    "import inspect; from safetensors import safe_open; "
    "raise SystemExit(0 if 'backend' in inspect.signature(safe_open).parameters else 1)"
)
_safe_check = _probe(_safe_code)
if _safe_check.returncode != 0:
    if _SAFETENSORS_LOADED_BEFORE_SETUP:
        raise RuntimeError(
            "Cần nâng cấp safetensors nhưng phiên hiện tại đã nạp thư viện này. "
            "Hãy Restart Session rồi chạy lại notebook từ đầu."
        )
    _pip_install("--upgrade", "--no-cache-dir", "safetensors>=0.8.0")
    _safe_check = _probe(_safe_code)
    if _safe_check.returncode != 0:
        raise RuntimeError(
            "Không cài được safetensors hỗ trợ backend=. Chi tiết: "
            + (_safe_check.stderr or _safe_check.stdout)[-2000:]
        )

_qwen35_code = (
    "import importlib.metadata; "
    "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
    "from transformers import AutoProcessor, AutoModelForMultimodalLM; "
    "assert 'qwen3_5' in CONFIG_MAPPING; "
    "print('transformers', importlib.metadata.version('transformers'), "
    "'| qwen3_5 + AutoModelForMultimodalLM: supported')"
)
_qwen35_check = _probe(_qwen35_code)
if _qwen35_check.returncode != 0:
    if _TRANSFORMERS_LOADED_BEFORE_SETUP:
        raise RuntimeError(
            "Transformers đã được nạp trong phiên này nhưng chưa hỗ trợ qwen3_5. "
            "Hãy Restart Session rồi chạy lại notebook từ đầu để cài đúng phiên bản."
        )
    print("Bản Transformers hiện tại chưa hỗ trợ Qwen3.5; đang thử bản PyPI mới nhất...")
    _pip_install("--upgrade", "--no-cache-dir", "transformers")
    _qwen35_check = _probe(_qwen35_code)
if _qwen35_check.returncode != 0:
    print("Bản PyPI chưa có đủ hỗ trợ Qwen3.5; đang thử mã nguồn chính thức của Hugging Face...")
    _pip_install(
        "--upgrade", "--no-cache-dir",
        "git+https://github.com/huggingface/transformers.git@main",
    )
    _qwen35_check = _probe(_qwen35_code)
if _qwen35_check.returncode != 0:
    raise RuntimeError(
        "Không cài được Transformers có hỗ trợ Qwen3.5. Chi tiết cài đặt: "
        + (_qwen35_check.stderr or _qwen35_check.stdout)[-2500:]
    )
print(_qwen35_check.stdout.strip())
print("safetensors hỗ trợ backend=.")


torchao: không dùng; cấu hình LoRA FP16 dùng Linear mặc định của PEFT.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 107.2 MB/s eta 0:00:00
Bản Transformers hiện tại chưa hỗ trợ Qwen3.5; đang thử bản PyPI mới nhất...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 297.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 380.0 MB/s eta 0:00:00
transformers 5.17.0 | qwen3_5 + AutoModelForMultimodalLM: supported
safetensors hỗ trợ backend=.


In [2]:
#@title Cấu hình, model gate và tìm dữ liệu chính thức
import os, re, json, math, random, hashlib, gc, inspect, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 20260817
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Để trống; thêm token có quyền ghi vào Kaggle Secrets với tên HF_TOKEN.
HF_TOKEN = ""  # Ch? ??c token t? Kaggle Secret t?n HF_TOKEN.

MODEL_ID = "Qwen/Qwen3.5-2B"
MODEL_ID_IN_BTC_PDF = True  # Đã đối chiếu PDF danh sách mô hình, trang 3.
TEAM_MODEL_APPROVED = True  # BTC đã xác nhận cho phép model này.
if not MODEL_ID_IN_BTC_PDF:
    raise RuntimeError("Model không nằm trong danh sách mô hình BTC.")
if not TEAM_MODEL_APPROVED:
    raise RuntimeError(
        "Chưa xác nhận Qwen/Qwen3.5-2B trong danh sách model đã đăng ký/được BTC duyệt cho đội. "
        "Đừng bật cờ này chỉ vì model có trong PDF."
    )

BASE_PIPELINE_PARAMS = 1_135_510_529  # Embedding + reranker của pipeline Task 2 hiện tại.
PARAM_CAP = 4_000_000_000
PARAM_RESERVE = 50_000_000  # Dự phòng cho selector/metadata học được khác trong pipeline.
TOP_CANDIDATES = 8
MAX_INPUT_TOKENS = 3072  # S18 inference/pipeline context
TRAIN_MAX_INPUT_TOKENS = 2048  # lower activation peak during LoRA training
MAX_NEW_TOKENS = 48
MAX_ANSWER_WORDS = 1400
DEV_SELECTION_LIMIT = 256
DEV_EVAL_LIMIT = None  # None = toàn bộ DEV.
OUT_DIR = Path("/kaggle/working/task2_qwen35_2b_selector_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROOT_HINTS = [
    Path("/kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test"),
    Path("/kaggle/input/uit-task2-publictest/LegalQA - Public Test"),
    Path.cwd() / "Task 2" / "LegalQA - Public Test",
    Path.cwd().parent / "Task 2" / "LegalQA - Public Test",
]
DATA_ROOT = next((p for p in ROOT_HINTS if (p / "train.json").is_file()), None)
if DATA_ROOT is None:
    roots = [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]
    train_hits = [
        p for root in roots if root.exists() for p in root.rglob("train.json")
        if "LegalQA - Public Test" in str(p)
    ]
    if not train_hits:
        raise FileNotFoundError("Không tìm thấy train.json của LegalQA - Public Test.")
    DATA_ROOT = train_hits[0].parent

ctx_files = sorted(DATA_ROOT.rglob("context_*.json"))
CTX_DIR = ctx_files[0].parent if ctx_files else None
if CTX_DIR is None:
    roots = [Path("/kaggle/input"), DATA_ROOT]
    ctx_files = sorted(
        p for root in roots if root.exists() for p in root.rglob("context_*.json")
    )
    CTX_DIR = ctx_files[0].parent if ctx_files else None
if CTX_DIR is None:
    raise FileNotFoundError("Không tìm thấy context_*.json trong corpus chính thức.")

from transformers.models.auto.configuration_auto import CONFIG_MAPPING
if "qwen3_5" not in CONFIG_MAPPING:
    raise RuntimeError(
        "Transformers chưa hỗ trợ qwen3_5. Chạy cell cài dependency đầu tiên; "
        "nếu vừa cài/nâng cấp trong phiên đã chạy, hãy Restart Session rồi chọn Run All."
    )
from transformers import AutoProcessor, AutoModelForMultimodalLM, Trainer, TrainingArguments

if not torch.cuda.is_available():
    raise RuntimeError("Cần GPU để fine-tune Qwen3.5-2B. Dùng Kaggle GPU 16GB.")
DEVICE = torch.device("cuda:0")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("Data:", DATA_ROOT)
print("Corpus:", CTX_DIR)
print("GPU:", torch.cuda.get_device_name(0), f"{GPU_GB:.1f} GiB")
print("Model:", MODEL_ID)
if GPU_GB < 14:
    raise RuntimeError("VRAM dưới 14 GiB; môi trường này không phù hợp cho LoRA FP16 của Qwen3.5-2B.")


Data: /kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test
Corpus: /kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test/selected-contexts/selected-contexts
GPU: Tesla T4 14.6 GiB
Model: Qwen/Qwen3.5-2B


In [3]:
#@title Đọc train và tạo split chống rò rỉ câu hỏi/citation
with open(DATA_ROOT / "train.json", encoding="utf-8") as f:
    raw_train = json.load(f)
TRAIN = {
    str(qid): {"question": str(row["question"]), "answer": str(row["answer"])}
    for qid, row in raw_train.items()
}
TRAIN_IDS = sorted(TRAIN)
assert len(TRAIN) == 7000, f"Số train khác dự kiến: {len(TRAIN)}"
assert len(ctx_files) == 8532, f"Số tài liệu corpus khác dự kiến: {len(ctx_files)}"

def norm_q(q):
    q = unicodedata.normalize("NFKC", str(q)).lower()
    return re.sub(r"\s+", " ", q).strip()

# Gộp câu hỏi trùng sau chuẩn hóa và câu có chung citation trong answer.
CITATION_RE = re.compile(
    r"\b(?:luật|bộ luật|nghị định|thông tư|nghị quyết|pháp lệnh|quyết định)"
    r"\s*(?:số\s*)?(\d+(?:/\d+){0,2}(?:/[a-zđ0-9.-]+)?)",
    flags=re.I,
)
parent = list(range(len(TRAIN_IDS)))
rank = [0] * len(TRAIN_IDS)
pos = {qid: i for i, qid in enumerate(TRAIN_IDS)}

def find_root(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find_root(a), find_root(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    if rank[ra] == rank[rb]:
        rank[ra] += 1

seen_keys = {}
for qid in TRAIN_IDS:
    i = pos[qid]
    keys = [("q", norm_q(TRAIN[qid]["question"]))]
    citations = {
        re.sub(r"\s+", "", m.group(1)).lower()
        for m in CITATION_RE.finditer(TRAIN[qid]["answer"])
    }
    keys.extend(("cite", c) for c in citations)
    for key in keys:
        if key in seen_keys:
            union(i, seen_keys[key])
        else:
            seen_keys[key] = i

GROUP_BY_QID = {qid: find_root(pos[qid]) for qid in TRAIN_IDS}
all_groups = np.array(sorted(set(GROUP_BY_QID.values())))
rng = np.random.default_rng(SEED + 31)
n_dev_groups = max(1, int(round(len(all_groups) * 0.15)))
dev_groups = set(rng.choice(all_groups, size=n_dev_groups, replace=False).tolist())
DEV_IDS = [qid for qid in TRAIN_IDS if GROUP_BY_QID[qid] in dev_groups]
FIT_IDS = [qid for qid in TRAIN_IDS if GROUP_BY_QID[qid] not in dev_groups]
assert set(FIT_IDS).isdisjoint(DEV_IDS)
assert set(FIT_IDS) | set(DEV_IDS) == set(TRAIN_IDS)
print(f"FIT={len(FIT_IDS)} DEV={len(DEV_IDS)} groups={len(all_groups)}")
print("question/citation groups disjoint:", True)


FIT=5938 DEV=1062 groups=4158
question/citation groups disjoint: True


In [4]:
#@title Chia corpus theo Điều và lập chỉ mục văn bản
WORD_WINDOW = 360
WORD_OVERLAP = 60
ARTICLE_RE = re.compile(r"(?=Điều\s+\d+[a-zđ]?\s*[.:])", flags=re.I)

def title_fallback(doc):
    title = str(doc.get("name") or doc.get("title") or "").strip()
    if title:
        return title
    slug = str(doc.get("link", "")).rstrip("/").split("/")[-1].split("?")[0]
    return re.sub(r"[-_]+", " ", slug)[:240]

def word_windows(text, size=WORD_WINDOW, overlap=WORD_OVERLAP):
    words = str(text).split()
    if not words:
        return []
    step = max(1, size - overlap)
    return [" ".join(words[s:s + size]) for s in range(0, len(words), step)]

def make_chunks():
    rows = []
    empty_docs = []
    for n, path in enumerate(ctx_files, 1):
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corpus JSON lỗi: {path}") from exc
        doc_id = str(doc.get("doc_id") or doc.get("id") or path.stem.replace("context_", "", 1))
        title = title_fallback(doc)
        passage = re.sub(
            r"\s+", " ", str(doc.get("passage") or doc.get("content") or doc.get("text") or "")
        ).strip()
        if not passage:
            empty_docs.append({"doc_id": doc_id, "file": path.name})
            continue
        sections = [s.strip() for s in ARTICLE_RE.split(passage) if s.strip()] or [passage]
        for sec_i, section in enumerate(sections):
            for part_i, text in enumerate(word_windows(section)):
                rows.append({
                    "doc_id": doc_id,
                    "parent_key": f"{doc_id}:{sec_i}",
                    "title": title,
                    "section": sec_i,
                    "part": part_i,
                    "text": text,
                    "retrieval_text": f"{title}\n{text}",
                })
        if n % 1000 == 0:
            print(f"{n}/{len(ctx_files)} documents | {len(rows):,} chunks", flush=True)
    return pd.DataFrame(rows), empty_docs

CH, EMPTY_DOCS = make_chunks()
EXPECTED_DOCUMENTS = len(ctx_files) - len(EMPTY_DOCS)
ACTUAL_DOCUMENTS = CH["doc_id"].nunique() if not CH.empty else 0
if ACTUAL_DOCUMENTS != EXPECTED_DOCUMENTS:
    raise RuntimeError(
        f"Corpus có {len(ctx_files)} file; đã bỏ qua {len(EMPTY_DOCS)} file không có nội dung, "
        f"nhưng chỉ còn {ACTUAL_DOCUMENTS} doc_id khác nhau (dự kiến {EXPECTED_DOCUMENTS}). "
        "Kiểm tra doc_id trùng hoặc tài liệu lỗi."
    )
if ACTUAL_DOCUMENTS == 0:
    raise RuntimeError("Corpus không có tài liệu chứa passage để lập chỉ mục.")
CH_TEXT = CH.retrieval_text.astype(str).tolist()
# Ghép các cửa sổ cùng Điều để giữ ngữ cảnh khi sinh nhãn bạc.
PARENT_TEXT = {}
PARENT_META = {}
for parent_key, group in CH.sort_values(["part"]).groupby("parent_key", sort=False):
    PARENT_TEXT[parent_key] = " ".join(group.text.astype(str))
    first = group.iloc[0]
    PARENT_META[parent_key] = {
        "doc_id": str(first.doc_id),
        "title": str(first.title),
    }
print(
    "chunks:", len(CH),
    "documents:", ACTUAL_DOCUMENTS,
    "empty source files skipped:", len(EMPTY_DOCS),
    "legal sections:", len(PARENT_TEXT),
)
if EMPTY_DOCS:
    print("Empty passage examples:", [row["file"] for row in EMPTY_DOCS[:5]])


1000/8532 documents | 38,803 chunks
2000/8532 documents | 77,670 chunks
3000/8532 documents | 115,984 chunks
4000/8532 documents | 157,114 chunks
5000/8532 documents | 194,741 chunks
6000/8532 documents | 233,117 chunks
7000/8532 documents | 275,433 chunks
8000/8532 documents | 317,457 chunks
chunks: 337355 documents: 8512 empty source files skipped: 20 legal sections: 171515
Empty passage examples: ['context_10533.json', 'context_131890.json', 'context_149317.json', 'context_177151.json', 'context_181693.json']


In [5]:
#@title Truy hồi 8 căn cứ ứng viên từ corpus BTC
from sklearn.feature_extraction.text import TfidfVectorizer

WORD_VEC = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=300_000,
    sublinear_tf=True, dtype=np.float32,
)
X_WORD = WORD_VEC.fit_transform(CH_TEXT).tocsr()
CH_PARENT = CH.parent_key.astype(str).tolist()

def retrieve_candidates(question, topk=TOP_CANDIDATES):
    qv = WORD_VEC.transform([str(question)])
    scores = (qv @ X_WORD.T).toarray().ravel()
    if not scores.size or float(scores.max()) <= 0.0:
        return []
    pool = min(len(scores), max(topk * 80, topk))
    part = np.argpartition(-scores, pool - 1)[:pool]
    order = part[np.argsort(-scores[part])]
    rows, seen_parents = [], set()
    for ix in order:
        parent_key = CH_PARENT[int(ix)]
        if parent_key in seen_parents:
            continue
        seen_parents.add(parent_key)
        meta = PARENT_META[parent_key]
        body = PARENT_TEXT[parent_key]
        head = meta["title"]
        rows.append({
            "parent_key": parent_key,
            "doc_id": meta["doc_id"],
            "head": head,
            "excerpt": body[:500],
            "answer_text": f"{head}\n{body}",
            "score": float(scores[int(ix)]),
        })
        if len(rows) >= topk:
            break
    return rows

# Xây candidate cache theo câu hỏi train; mọi candidate đều lấy từ corpus chính thức.
CANDIDATES = {}
for n, qid in enumerate(TRAIN_IDS, 1):
    CANDIDATES[qid] = retrieve_candidates(TRAIN[qid]["question"])
    if n % 500 == 0:
        print(f"{n}/{len(TRAIN_IDS)} queries", flush=True)
print("candidate sets:", len(CANDIDATES),
      "| empty:", sum(not rows for rows in CANDIDATES.values()))


500/7000 queries
1000/7000 queries
1500/7000 queries
2000/7000 queries
2500/7000 queries
3000/7000 queries
3500/7000 queries
4000/7000 queries
4500/7000 queries
5000/7000 queries
5500/7000 queries
6000/7000 queries
6500/7000 queries
7000/7000 queries
candidate sets: 7000 | empty: 0


In [6]:
#@title Tạo prompt S18 và nhãn bạc tối ưu METEOR
import nltk
from nltk.translate.meteor_score import meteor_score

# Scorer BTC dùng đúng meteor_score([ref.split()], pred.split()).
for resource in ("wordnet", "omw-1.4"):
    try:
        nltk.data.find(f"corpora/{resource}")
    except LookupError:
        # Cùng resource mà scorer chính thức của BTC tải để tính METEOR.
        nltk.download(resource, quiet=True)

QWEN_SYS = (
    "Bạn là bộ chọn căn cứ pháp luật Việt Nam. "
    "Chỉ chọn các mục cần thiết để trả lời câu hỏi. "
    "Không giải thích và không chép lại nội dung điều luật."
)

def build_user_prompt(question, rows):
    if not rows:
        options = "(Không tìm thấy căn cứ trong corpus.)"
    else:
        options = "\n\n".join(
            f"[{j + 1}] {row['head']}\n{row['excerpt']}"
            for j, row in enumerate(rows)
        )
    return (
        f"CÂU HỎI:\n{question}\n\nCÁC CĂN CỨ ỨNG VIÊN:\n{options}\n\n"
        "Trả lời đúng định dạng <selection>1,2</selection>. "
        "Chọn tối đa 3 số theo độ liên quan giảm dần."
    )

def official_meteor(reference, prediction):
    ref = reference.split() if isinstance(reference, str) else list(reference)
    pred = prediction.split() if isinstance(prediction, str) else list(prediction)
    if not pred:
        return 0.0
    return float(meteor_score([ref], pred))

def selected_answer(rows, picked, max_words=MAX_ANSWER_WORDS):
    blocks, word_count = [], 0
    for i in picked:
        if i < 0 or i >= len(rows) or word_count >= max_words:
            continue
        words = rows[i]["answer_text"].split()
        take = words[:max_words - word_count]
        if take:
            blocks.append(" ".join(take))
            word_count += len(take)
    return "\n".join(blocks)

def silver_selection(rows, gold):
    if not rows:
        return [], 0.0
    best_single = max(
        range(len(rows)),
        key=lambda i: official_meteor(gold, selected_answer(rows, [i])),
    )
    chosen = [best_single]
    current_score = official_meteor(gold, selected_answer(rows, chosen))
    remaining = [i for i in range(len(rows)) if i != best_single]
    for _ in range(2):
        best_i, best_score = None, current_score
        for i in remaining:
            trial_ids = sorted(chosen + [i])
            score = official_meteor(gold, selected_answer(rows, trial_ids))
            if score > best_score + 1e-8:
                best_i, best_score = i, score
        if best_i is None:
            break
        chosen.append(best_i)
        remaining.remove(best_i)
        chosen.sort()
        current_score = best_score
    return chosen, current_score

def make_examples(qids):
    out = []
    for qid in qids:
        rows = CANDIDATES[qid]
        if not rows:
            continue
        picked, silver_score = silver_selection(rows, TRAIN[qid]["answer"])
        target = "<selection>" + ",".join(str(i + 1) for i in picked) + "</selection>"
        out.append({
            "qid": qid,
            "prompt": build_user_prompt(TRAIN[qid]["question"], rows),
            "target": target,
            "candidates": rows,
            "gold": TRAIN[qid]["answer"],
            "silver_meteor": silver_score,
        })
    return out

FIT_EXAMPLES = make_examples(FIT_IDS)
DEV_EXAMPLES = make_examples(DEV_IDS)
assert FIT_EXAMPLES and DEV_EXAMPLES
print("SFT examples:", len(FIT_EXAMPLES), "FIT /", len(DEV_EXAMPLES), "DEV")
print("mean silver METEOR:",
      float(np.mean([x["silver_meteor"] for x in FIT_EXAMPLES])))
print("sample target:", FIT_EXAMPLES[0]["target"])
print("sample candidate count:", len(FIT_EXAMPLES[0]["candidates"]))


SFT examples: 5938 FIT / 1062 DEV
mean silver METEOR: 0.533751859421836
sample target: <selection>1</selection>
sample candidate count: 8


In [7]:
#@title Qwen3.5-compatible processor and base-model baseline
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=False)
tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
if tokenizer.eos_token_id is None:
    raise RuntimeError("Tokenizer thiếu eos_token_id.")

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
).to(DEVICE).eval()

BASE_MODEL_PARAMS = sum(int(p.numel()) for p in model.parameters())
print(f"Qwen3.5-2B logical parameters: {BASE_MODEL_PARAMS:,}")
for cfg in (model.config, getattr(model.config, "text_config", None)):
    if cfg is not None and hasattr(cfg, "use_cache"):
        cfg.use_cache = True

def render_chat(user_prompt):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
    ]
    try:
        return processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

def parse_selection(text, rows):
    match = re.search(r"<selection>\s*([^<]+)\s*</selection>", str(text), flags=re.I)
    if match:
        candidate = match.group(1)
    else:
        candidate = str(text).strip().splitlines()[-1] if str(text).strip() else ""
    numbers = [int(x) for x in re.findall(r"\d+", candidate)]
    picked = [n - 1 for n in dict.fromkeys(numbers) if 1 <= n <= len(rows)][:3]
    return picked or list(range(min(2, len(rows))))

def generate_selection(model_to_use, example):
    prompt = render_chat(example["prompt"])
    inputs = processor(
        text=[prompt], padding=True, truncation=True, max_length=MAX_INPUT_TOKENS,
        return_tensors="pt",
    ).to(DEVICE)
    with torch.inference_mode():
        output = model_to_use.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0, prompt_len:], skip_special_tokens=True).strip()

def evaluate_selector(model_to_use, examples, tag):
    rows = examples if DEV_EVAL_LIMIT is None else examples[:DEV_EVAL_LIMIT]
    model_to_use.eval()
    meteor_values = []
    for n, ex in enumerate(rows, 1):
        text = generate_selection(model_to_use, ex)
        picked = parse_selection(text, ex["candidates"])
        answer = selected_answer(ex["candidates"], picked)
        meteor_values.append(official_meteor(ex["gold"], answer))
        if n % 100 == 0:
            print(f"{tag}: {n}/{len(rows)}", flush=True)
    result = {"n": len(rows), "meteor_proxy": float(np.mean(meteor_values))}
    print(tag, result)
    return result

BASE_DEV_METRICS = evaluate_selector(model, DEV_EXAMPLES, "base Qwen on DEV")


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Qwen3.5-2B logical parameters: 2,213,241,664


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


base Qwen on DEV: 100/1062
base Qwen on DEV: 200/1062
base Qwen on DEV: 300/1062
base Qwen on DEV: 400/1062
base Qwen on DEV: 500/1062
base Qwen on DEV: 600/1062
base Qwen on DEV: 700/1062
base Qwen on DEV: 800/1062
base Qwen on DEV: 900/1062
base Qwen on DEV: 1000/1062
base Qwen on DEV {'n': 1062, 'meteor_proxy': 0.41943988720965625}


In [8]:
#@title Gắn LoRA vào các linear layer của text decoder, kiểm tra ngân sách
# Gỡ backend torchao tùy chọn trước khi PEFT chọn dispatcher. Có thể chạy lại
# cell này ngay trong phiên cũ sau lỗi; khi đó làm mới bộ kiểm tra phiên bản của PEFT.
import importlib, subprocess, sys
_torchao_probe_code = (
    "import importlib.util; "
    "raise SystemExit(0 if importlib.util.find_spec('torchao') is not None else 1)"
)
_torchao_check = subprocess.run(
    [sys.executable, "-c", _torchao_probe_code], capture_output=True, text=True
)
if _torchao_check.returncode == 0:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
        check=True,
    )
    if "peft.import_utils" in sys.modules:
        importlib.reload(sys.modules["peft.import_utils"])
    if "peft.tuners.lora.torchao" in sys.modules:
        importlib.reload(sys.modules["peft.tuners.lora.torchao"])
    _torchao_check = subprocess.run(
        [sys.executable, "-c", _torchao_probe_code], capture_output=True, text=True
    )
    if _torchao_check.returncode == 0:
        raise RuntimeError("Không gỡ được torchao cũ; PEFT chưa thể gắn LoRA an toàn.")

from torch import nn
from peft import LoraConfig, get_peft_model

LORA_SUFFIXES = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "qkv_proj", "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj",
    "gate_proj", "up_proj", "down_proj",
}
EXCLUDE_MARKERS = ("vision", "visual", "image", "projector", "merger")
TARGET_MODULES = [
    name for name, module in model.named_modules()
    if isinstance(module, nn.Linear)
    and name.rsplit(".", 1)[-1] in LORA_SUFFIXES
    and ".layers." in name
    and not any(marker in name.lower() for marker in EXCLUDE_MARKERS)
]
if not TARGET_MODULES:
    sample = [name for name, module in model.named_modules() if isinstance(module, nn.Linear)][:60]
    raise RuntimeError(
        "Không tìm thấy linear layer phù hợp cho LoRA. Kiểm tra kiến trúc Qwen3.5 đang cài. "
        f"Linear layer mẫu: {sample}"
    )
print("LoRA targets:", len(TARGET_MODULES))
print("Target sample:", TARGET_MODULES[:8])

for cfg in (model.config, getattr(model.config, "text_config", None)):
    if cfg is not None and hasattr(cfg, "use_cache"):
        cfg.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ADAPTER_PARAMS = sum(
    int(p.numel()) for name, p in model.named_parameters()
    if p.requires_grad or "lora_" in name
)
SYSTEM_PARAMS = BASE_PIPELINE_PARAMS + BASE_MODEL_PARAMS + ADAPTER_PARAMS
SYSTEM_PARAMS_WITH_RESERVE = SYSTEM_PARAMS + PARAM_RESERVE
print(f"existing Task 2 embedding+reranker: {BASE_PIPELINE_PARAMS:,}")
print(f"Qwen3.5-2B base: {BASE_MODEL_PARAMS:,}")
print(f"LoRA adapter: {ADAPTER_PARAMS:,}")
print(f"system total: {SYSTEM_PARAMS:,}; plus reserve: {SYSTEM_PARAMS_WITH_RESERVE:,}")
assert SYSTEM_PARAMS_WITH_RESERVE < PARAM_CAP, "Tổng tham số vượt hoặc sát ngưỡng 4B."
model.config.use_cache = False


LoRA targets: 186
Target sample: ['model.language_model.layers.0.linear_attn.out_proj', 'model.language_model.layers.0.linear_attn.in_proj_qkv', 'model.language_model.layers.0.linear_attn.in_proj_z', 'model.language_model.layers.0.linear_attn.in_proj_b', 'model.language_model.layers.0.linear_attn.in_proj_a', 'model.language_model.layers.0.mlp.gate_proj', 'model.language_model.layers.0.mlp.up_proj', 'model.language_model.layers.0.mlp.down_proj']
trainable params: 8,409,600 || all params: 2,221,651,264 || trainable%: 0.3785
existing Task 2 embedding+reranker: 1,135,510,529
Qwen3.5-2B base: 2,213,241,664
LoRA adapter: 8,409,600
system total: 3,357,161,793; plus reserve: 3,407,161,793


In [9]:
#@title Completion-only dataset, METEOR-aware evaluation and Trainer
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

def chat_prompt_ids(user_prompt):
    rendered = render_chat(user_prompt)
    return tokenizer(rendered, add_special_tokens=False)["input_ids"]

class SelectionDataset(Dataset):
    def __init__(self, examples, max_input=MAX_INPUT_TOKENS):
        self.rows = []
        for ex in examples:
            prompt_ids = chat_prompt_ids(ex["prompt"])
            target_ids = tokenizer(ex["target"], add_special_tokens=False)["input_ids"]
            target_ids = target_ids + [tokenizer.eos_token_id]
            if len(target_ids) >= max_input:
                target_ids = target_ids[:max_input - 1] + [tokenizer.eos_token_id]
            prompt_budget = max(1, max_input - len(target_ids))
            if len(prompt_ids) > prompt_budget:
                # Keep system/question prefix and the end of candidates.
                head_budget = min(512, max(1, prompt_budget // 3))
                tail_budget = max(1, prompt_budget - head_budget)
                prompt_ids = prompt_ids[:head_budget] + prompt_ids[-tail_budget:]
            ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids
            self.rows.append({"input_ids": ids, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

class SelectionCollator:
    def __init__(self, tok):
        self.tok = tok

    def __call__(self, batch):
        width = max(len(row["input_ids"]) for row in batch)
        input_ids, attention, labels = [], [], []
        for row in batch:
            pad_n = width - len(row["input_ids"])
            input_ids.append(row["input_ids"] + [self.tok.pad_token_id] * pad_n)
            attention.append([1] * len(row["input_ids"]) + [0] * pad_n)
            labels.append(row["labels"] + [-100] * pad_n)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

rng = np.random.default_rng(SEED + 77)
n_select = min(DEV_SELECTION_LIMIT, len(DEV_EXAMPLES))
select_idx = np.sort(rng.choice(len(DEV_EXAMPLES), size=n_select, replace=False))
DEV_SELECTION_EXAMPLES = [DEV_EXAMPLES[int(i)] for i in select_idx]

train_ds = SelectionDataset(FIT_EXAMPLES, max_input=TRAIN_MAX_INPUT_TOKENS)
eval_ds = SelectionDataset(DEV_SELECTION_EXAMPLES, max_input=TRAIN_MAX_INPUT_TOKENS)
collator = SelectionCollator(tokenizer)
print("tokenized rows:", len(train_ds), "FIT /", len(eval_ds), "DEV checkpoint-selection rows")

def selection_metric(eval_pred):
    predictions = eval_pred.predictions
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    pred_texts = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    values = []
    for ex, text in zip(DEV_SELECTION_EXAMPLES, pred_texts):
        picked = parse_selection(text, ex["candidates"])
        answer = selected_answer(ex["candidates"], picked)
        values.append(official_meteor(ex["gold"], answer))
    return {"meteor": float(np.mean(values)) if values else 0.0}

class MeteorSelectionTrainer(Trainer):
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels")
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss.detach().mean() if getattr(outputs, "loss", None) is not None else None
            if prediction_loss_only:
                return loss, None, None
            # Eval inputs contain gold target tokens for loss. Generate from the
            # prompt prefix only; otherwise the model sees the answer label.
            prompt_width = int(labels[0].eq(-100).sum().item())
            generation_inputs = {
                "input_ids": inputs["input_ids"][:, :prompt_width],
                "attention_mask": inputs["attention_mask"][:, :prompt_width],
            }
            generated = model.generate(
                **generation_inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        predictions = generated[:, prompt_width:]
        target_rows = [row[row.ne(-100)] for row in labels]
        target_width = max((len(row) for row in target_rows), default=1)
        target_ids = torch.full(
            (len(target_rows), target_width), -100, dtype=torch.long, device=predictions.device
        )
        for i, row in enumerate(target_rows):
            target_ids[i, :len(row)] = row
        return loss, predictions.detach(), target_ids

train_sig = inspect.signature(TrainingArguments.__init__).parameters
eval_strategy_arg = "eval_strategy" if "eval_strategy" in train_sig else "evaluation_strategy"
training_kwargs = {
    "output_dir": str(OUT_DIR / "checkpoints"),
    "num_train_epochs": 2,
    "learning_rate": 8e-5,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "fp16": True,
    "gradient_checkpointing": True,
    "optim": "adamw_torch",
    "save_strategy": "steps",
    "save_steps": 25,
    "save_total_limit": 2,
    "logging_steps": 25,
    "report_to": "none",
    "remove_unused_columns": False,
    "load_best_model_at_end": True,
    "metric_for_best_model": "meteor",
    "greater_is_better": True,
    "seed": SEED,
    "data_seed": SEED,
}
if "gradient_checkpointing_kwargs" in train_sig:
    training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

WARMUP_RATIO = 0.03
if "warmup_ratio" in train_sig:
    training_kwargs["warmup_ratio"] = WARMUP_RATIO
elif "warmup_steps" in train_sig:
    micro_batches_per_epoch = math.ceil(
        len(train_ds) / training_kwargs["per_device_train_batch_size"]
    )
    updates_per_epoch = math.ceil(
        micro_batches_per_epoch / training_kwargs["gradient_accumulation_steps"]
    )
    total_update_steps = updates_per_epoch * math.ceil(training_kwargs["num_train_epochs"])
    training_kwargs["warmup_steps"] = max(1, math.ceil(total_update_steps * WARMUP_RATIO))
    print(
        f"Transformers không nhận warmup_ratio; dùng warmup_steps="
        f"{training_kwargs['warmup_steps']} trên {total_update_steps} bước."
    )
else:
    raise RuntimeError(
        "TrainingArguments của Transformers hiện tại không hỗ trợ warmup_ratio "
        "hoặc warmup_steps."
    )
training_kwargs[eval_strategy_arg] = "steps"
training_kwargs["eval_steps"] = 25
unsupported_args = sorted(set(training_kwargs) - set(train_sig))
if unsupported_args:
    raise RuntimeError(
        "Các tham số TrainingArguments không được phiên bản Transformers hỗ trợ: "
        + str(unsupported_args)
    )
args = TrainingArguments(**training_kwargs)
trainer = MeteorSelectionTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    compute_metrics=selection_metric,
)
print("Checkpoint được chọn theo eval_meteor sinh trên DEV selection subset.")


tokenized rows: 5938 FIT / 256 DEV checkpoint-selection rows
Transformers không nhận warmup_ratio; dùng warmup_steps=23 trên 744 bước.
Checkpoint được chọn theo eval_meteor sinh trên DEV selection subset.


In [10]:
#@title Fine-tune và lưu adapter tốt nhất
# Clear cached blocks left by the base-model DEV generation pass.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    free_gb, total_gb = torch.cuda.mem_get_info()
    print(f"VRAM free before train: {free_gb / 1024**3:.2f} / {total_gb / 1024**3:.2f} GiB")
from transformers.trainer_utils import get_last_checkpoint
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
RESUME_CHECKPOINT = get_last_checkpoint(str(CHECKPOINT_DIR))
if RESUME_CHECKPOINT:
    print("Resuming from checkpoint:", RESUME_CHECKPOINT)
else:
    print("No checkpoint found; starting training from the beginning.")
trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
BEST_ADAPTER_DIR = OUT_DIR / "adapter_best"
trainer.save_model(str(BEST_ADAPTER_DIR))
processor.save_pretrained(BEST_ADAPTER_DIR)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval METEOR:", trainer.state.best_metric)
print("Adapter:", BEST_ADAPTER_DIR)


VRAM free before train: 10.24 / 14.56 GiB


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
#@title So sánh adapter với base trên cùng DEV và ghi manifest
TUNED_DEV_METRICS = evaluate_selector(model, DEV_EXAMPLES, "LoRA best on DEV")
delta = TUNED_DEV_METRICS["meteor_proxy"] - BASE_DEV_METRICS["meteor_proxy"]
print("proxy METEOR delta:", round(delta, 6))
if delta <= 0:
    print("Không bật adapter vào pipeline chính: proxy DEV chưa vượt base Qwen.")
else:
    print("Adapter là ứng viên; vẫn cần chạy scoring end-to-end bằng pipeline Task 2 chính.")

manifest = {
    "task": "Task 2 LegalQA evidence selector",
    "base_model": MODEL_ID,
    "model_list_source": "[DSC@UIT 2026] Danh sách mô hình.pdf, page 3",
    "team_model_approval_confirmed": bool(TEAM_MODEL_APPROVED),
    "base_model_parameters": int(BASE_MODEL_PARAMS),
    "adapter_parameters": int(ADAPTER_PARAMS),
    "existing_embedding_reranker_parameters": int(BASE_PIPELINE_PARAMS),
    "system_parameters": int(SYSTEM_PARAMS),
    "parameter_reserve": int(PARAM_RESERVE),
    "system_parameters_with_reserve": int(SYSTEM_PARAMS_WITH_RESERVE),
    "parameter_cap": int(PARAM_CAP),
    "seed": int(SEED),
    "fit_questions": int(len(FIT_EXAMPLES)),
    "dev_questions": int(len(DEV_EXAMPLES)),
    "dev_checkpoint_selection_questions": int(len(DEV_SELECTION_EXAMPLES)),
    "split": "normalized question + shared answer citations, connected components",
    "candidate_source": "official Task 2 corpus only; TF-IDF word 1-2 gram",
    "silver_labels": "greedy subset maximizing official METEOR against official FIT answers",
    "official_data_only": True,
    "manual_labels": False,
    "external_augmentation": False,
    "external_api": False,
    "base_dev_proxy_meteor": float(BASE_DEV_METRICS["meteor_proxy"]),
    "tuned_dev_proxy_meteor": float(TUNED_DEV_METRICS["meteor_proxy"]),
    "dev_proxy_delta": float(delta),
    "best_checkpoint": str(trainer.state.best_model_checkpoint),
    "adapter_dir": str(BEST_ADAPTER_DIR),
}
manifest_path = OUT_DIR / "manifest_task2_qwen35_2b_selector.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

# Tích hợp sau khi proxy và METEOR end-to-end đều thắng:
# from peft import PeftModel
# base = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
# tuned = PeftModel.from_pretrained(base, str(BEST_ADAPTER_DIR)).eval()
# Trong qwen_select_shard của notebook chính, nạp adapter trước khi gọi generate().
# Chỉ bật S18 và thêm model vào manifest/allowlist khi model đã được BTC duyệt
# và tổng tham số hệ thống Task 2 vẫn dưới 4B.


In [ ]:
#@title (Tùy chọn) Tải adapter và manifest lên Hugging Face
# Mặc định bật tự tải lên sau khi huấn luyện; đặt False nếu chỉ muốn huấn luyện.
UPLOAD_TO_HF = True  # Tự tải lên sau khi huấn luyện; đặt False nếu chỉ muốn huấn luyện.
HF_REPO_NAME = "dsc2026-task2-qwen35-2b-selector-lora"
HF_REPO_PRIVATE = True

if UPLOAD_TO_HF:
    hf_token = (HF_TOKEN or os.environ.get("HF_TOKEN") or "").strip()
    if not hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError(
            "Hãy tạo Kaggle Secret tên HF_TOKEN bằng token Hugging Face có quyền ghi."
        )

    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    owner = api.whoami()["name"]
    HF_REPO_ID = f"{owner}/{HF_REPO_NAME}"
    api.create_repo(
        repo_id=HF_REPO_ID, repo_type="model", private=HF_REPO_PRIVATE, exist_ok=True
    )
    # Chỉ tải adapter LoRA và processor/tokenizer đi kèm, không tải lại model gốc 2B.
    api.upload_folder(
        repo_id=HF_REPO_ID, repo_type="model", folder_path=str(BEST_ADAPTER_DIR)
    )
    api.upload_file(
        path_or_fileobj=str(manifest_path),
        path_in_repo="manifest_task2_qwen35_2b_selector.json",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print("Uploaded private adapter repo:", HF_REPO_ID)
else:
    print("HF upload skipped. Set UPLOAD_TO_HF=True after training to upload the adapter.")
